In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from utils.skeletonPATS import SkeletonPATS
from pats.utils import load_multiple_samples, get_speaker_intervals, get_all_missing_intervals
import numpy as np
from typing import List, Union

In [3]:
class SkeletonDataset(Dataset):
    def __init__(self, speakers :Union[List[str], str] ="fallon", split="train"):
        """
        data: tensor (N, 52, 2) o numpy array equivalente
        """
        skeleton_poses = []
        self.speakers = list(speakers) if isinstance(speakers, str) else speakers
        for speaker in self.speakers:
            intervals = get_speaker_intervals(speaker=speaker, split=split)
        
            clips = load_multiple_samples(
                speaker=speaker, 
                interval_ids=intervals
            )
            for clip in clips:
                pose = clip['pose']  # (N, 52, 2)
                pose[:,0] = [0.0, 0.0]
                skeleton_poses.append(pose)

        skeleton_poses = np.concatenate(skeleton_poses, axis=0)
        skeleton_poses = SkeletonPATS.normalize_skeleton(skeleton_poses)

        skeleton_poses = torch.tensor(skeleton_poses, dtype=torch.float32)
        self.data = skeleton_poses



    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx]
        


In [4]:
dataset = SkeletonDataset(speakers=["fallon", "seth", "shelly"], split="dev")
print(f"{len(dataset)} samples loaded.")

193769 samples loaded.


In [6]:
fallon = SkeletonDataset(speakers=["fallon"], split="dev")
seth = SkeletonDataset(speakers=["seth"], split="dev")
shelly = SkeletonDataset(speakers=["shelly"], split="dev")

print(f"Fallon: {len(fallon)} samples loaded.")
print(f"Seth: {len(seth)} samples loaded.")
print(f"Shelly: {len(shelly)} samples loaded.")

Fallon: 46335 samples loaded.
Seth: 48121 samples loaded.
Shelly: 99313 samples loaded.
